# Construct Demographic-Conditioned Mobility Behaviour Profile

## Overview

This notebook connects the Representative Population Layer with the Behavioural Evidence Layer of the Human Mobility Representation Framework.

The Representative Population Pool was initially constructed from the Cartesian product of the selected demographic attribute categories. As this process produced all possible combinations, logical consistency rules were applied to remove infeasible profiles, such as assigning an occupation or industry to a person who is not employed. The resulting logically constrained population pool therefore contains feasible representative combinations of demographic and socio-economic attributes.

Given one representative profile from this population pool, this notebook retrieves the corresponding trip purpose distribution based on age and gender. Each trip purpose is then associated with its temporal and travel mode distributions derived from the National Travel Survey.

The output is a complete Demographic-Conditioned Mobility Behaviour Profile that retains all supported trip purposes and their associated probabilities. It represents a probabilistic behavioural space rather than a sampled or realised journey and provides the input for the subsequent Behaviour-to-POI Integration.


## 1. Import Libraries

In [1]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

## 2. Set Up File Paths

In [2]:
project_root = Path.cwd().parents[1]

population_file = (
    project_root
    / "output"
    / "demographic_profiles"
    / "representative_population.parquet"
)

behavioural_evidence_file = (
    project_root
    / "output"
    / "behavioural_evidence"
    / "nts_behavioural_evidence.json"
)

## 3. Load Input Data

In [3]:
population = pd.read_parquet(population_file)

with open(behavioural_evidence_file, "r", encoding="utf-8") as file:
    behavioural_evidence = json.load(file)

In [4]:
with open(behavioural_evidence_file, "r", encoding="utf-8") as file:
    behavioural_evidence = json.load(file)

## 4. Inspect Input Data

In [5]:
print(f"Population shape: {population.shape}")
print(f"Population columns: {population.columns.tolist()}")

print(f"\nBehavioural evidence type: {type(behavioural_evidence).__name__}")
print(f"Number of trip purposes: {len(behavioural_evidence)}")

Population shape: (2869120, 14)
Population columns: ['Age', 'Gender', 'Household Type', 'Household Count', 'Economic Activity Status', 'Economic Count', 'Industry', 'Industry Count', 'Ethnic Group', 'Ethnic Count', 'Distance travelled to work', 'Travel Distance Count', 'Car Ownership', 'Car Ownership Count']

Behavioural evidence type: dict
Number of trip purposes: 14


In [6]:
display(population.head())

,Age,Gender,Household Type,Household Count,Economic Activity Status,Economic Count,Industry,Industry Count,Ethnic Group,Ethnic Count,Distance travelled to work,Travel Distance Count,Car Ownership,Car Ownership Count
0,0-4,Female,Couple household: With dependent children,20814,Does not apply,34378,Does not apply,34378,"Asian, Asian British or Asian Welsh: Bangladeshi",833,Does not apply,34378,1 car or van in household,17962
1,0-4,Female,Couple household: With dependent children,20814,Does not apply,34378,Does not apply,34378,"Asian, Asian British or Asian Welsh: Bangladeshi",833,Does not apply,34378,2 cars or vans in household,11303
2,0-4,Female,Couple household: With dependent children,20814,Does not apply,34378,Does not apply,34378,"Asian, Asian British or Asian Welsh: Bangladeshi",833,Does not apply,34378,3 or more cars or vans in household,1499
3,0-4,Female,Couple household: With dependent children,20814,Does not apply,34378,Does not apply,34378,"Asian, Asian British or Asian Welsh: Bangladeshi",833,Does not apply,34378,No cars or vans in household,5428
4,0-4,Female,Couple household: With dependent children,20814,Does not apply,34378,Does not apply,34378,"Black, Black British, Black Welsh, Caribbean or African: African",3717,Does not apply,34378,1 car or van in household,17962


In [7]:
print("Available trip purposes:")
print(list(behavioural_evidence.keys()))

Available trip purposes:
['Commuting', 'Business', 'Education', 'Escort Education', 'Shopping', 'Other Escort', 'Personal Business', 'Visiting Friends at Private Home', 'Visiting Friends Elsewhere', 'Entertainment or Public Activity', 'Sport: Participate', 'Holiday: Base', 'Day Trip', 'Just Walk']


In [8]:
example_purpose = "Commuting"

print(f"Evidence components for {example_purpose}:")
print(behavioural_evidence[example_purpose].keys())

Evidence components for Commuting:
dict_keys(['baseline_profile', 'travel_mode_profile', 'temporal_pattern_profile'])


In [9]:
representative_profile = population.iloc[0]

display(representative_profile.to_frame(name="Value"))

,Value
Age,0-4
Gender,Female
Household Type,Couple household: With dependent children
Household Count,20814
Economic Activity Status,Does not apply
Economic Count,34378
Industry,Does not apply
Industry Count,34378
Ethnic Group,"Asian, Asian British or Asian Welsh: Bangladeshi"
Ethnic Count,833


## 5. Prepare and Validate Demographic Lookup Values

The trip purpose baseline distributions are conditioned on age and gender.
The corresponding values are therefore extracted from the selected
representative profile and used as lookup attributes within the Behavioural
Evidence Layer.

The demographic labels have been harmonised across the Representative
Population Layer and the Behavioural Evidence Layer. Both layers use the
`Gender` attribute with the standardised categories `Female` and `Male`;
therefore, no additional category mapping is required at this integration
stage.

In [10]:
representative_age = representative_profile["Age"]
representative_gender = representative_profile["Gender"]

behavioural_age = representative_age
behavioural_gender = representative_gender

print("Representative population values:")
print(f"  Age: {representative_age}")
print(f"  Gender: {representative_gender}")

print("\nBehavioural evidence lookup values:")
print(f"  Age: {behavioural_age}")
print(f"  Gender: {behavioural_gender}")

Representative population values:
  Age: 0-4
  Gender: Female

Behavioural evidence lookup values:
  Age: 0-4
  Gender: Female


## 6. Retrieve Trip Purpose Probabilities

The trip purpose distribution is conditioned on the age and gender of the
representative profile. For each trip purpose, the corresponding baseline
probability is retrieved from the behavioural evidence.

A single trip purpose is initially inspected to validate the demographic
matching process before applying the lookup to all trip purposes.

In [11]:
example_purpose = "Commuting"

baseline_profile = behavioural_evidence[
    example_purpose
]["baseline_profile"]

matching_baseline = [
    record
    for record in baseline_profile
    if record["Age"] == behavioural_age
    and record["Gender"] == behavioural_gender
]

if len(matching_baseline) == 0:
    raise ValueError(
        "No trip purpose probability found for "
        f"purpose={example_purpose}, "
        f"age={behavioural_age}, "
        f"gender={behavioural_gender}."
    )

if len(matching_baseline) > 1:
    raise ValueError(
        "Multiple trip purpose probabilities found for "
        f"purpose={example_purpose}, "
        f"age={behavioural_age}, "
        f"gender={behavioural_gender}."
    )

matching_baseline[0]

{'Year': 2024,
 'Gender': 'Female',
 'Age': '0-4',
 'Trip Purpose Probability': 0.0036156239}

Following validation of the demographic lookup, the same process is applied
to all unified trip purposes. This produces the complete trip purpose
distribution conditioned on the age and gender of the selected representative
profile.

Because the framework assumes that a trip has occurred, the probabilities
across all mutually exclusive trip purposes are expected to sum to one.

In [12]:
trip_purpose_probabilities = {}

for purpose, evidence in behavioural_evidence.items():
    baseline_profile = evidence["baseline_profile"]

    matching_records = [
        record
        for record in baseline_profile
        if record["Age"] == behavioural_age
        and record["Gender"] == behavioural_gender
    ]

    if len(matching_records) == 0:
        raise ValueError(
            "No trip purpose probability found for "
            f"purpose={purpose}, "
            f"age={behavioural_age}, "
            f"gender={behavioural_gender}."
        )

    if len(matching_records) > 1:
        raise ValueError(
            "Multiple trip purpose probabilities found for "
            f"purpose={purpose}, "
            f"age={behavioural_age}, "
            f"gender={behavioural_gender}."
        )

    trip_purpose_probabilities[purpose] = matching_records[0][
        "Trip Purpose Probability"
    ]

In [13]:
trip_purpose_distribution = pd.DataFrame(
    [
        {
            "Trip Purpose": purpose,
            "Probability": probability,
        }
        for purpose, probability
        in trip_purpose_probabilities.items()
    ]
).sort_values(
    "Probability",
    ascending=False,
    ignore_index=True,
)

display(trip_purpose_distribution)

,Trip Purpose,Probability
0,Education,0.327535
1,Other Escort,0.142315
2,Visiting Friends at Private Home,0.106521
3,Entertainment or Public Activity,0.078185
4,Personal Business,0.075573
5,Just Walk,0.053662
6,Escort Education,0.044123
7,Visiting Friends Elsewhere,0.043625
8,Day Trip,0.041132
9,Sport: Participate,0.035980


In [14]:
trip_purpose_probability_sum = (
    trip_purpose_distribution["Probability"].sum()
)

print(
    "Trip purpose probability sum: "
    f"{trip_purpose_probability_sum:.10f}"
)

assert trip_purpose_distribution["Probability"].notna().all(), (
    "Missing trip purpose probabilities detected."
)

assert trip_purpose_distribution["Probability"].between(0, 1).all(), (
    "Trip purpose probabilities must be between 0 and 1."
)

assert abs(trip_purpose_probability_sum - 1.0) < 1e-8, (
    "Trip purpose probabilities do not sum to 1."
)

print("Trip purpose distribution validation: PASS")

Trip purpose probability sum: 0.9999999999
Trip purpose distribution validation: PASS
